# 🧠 RAG Playground: Step-by-Step RAG Mechanics

Welcome! This notebook allows you to experiment with RAG (Retrieval-Augmented Generation) mechanics step-by-step. By running these cells inside VS Code, you will see exactly how:
1. Text is converted into mathematical vectors (Embeddings).
2. Vectors are stored and queried in a Vector Database (ChromaDB).
3. Chunks are retrieved and passed to Gemini with a custom prompt to generate the audited report.

In [ ]:
# 1. Setup & Imports
import os
import json
import chromadb
from google import genai
from dotenv import load_dotenv

# Load API Keys from backend/.env
load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")

if api_key:
    client = genai.Client(api_key=api_key)
    print("✅ Gemini client initialized successfully!")
else:
    print("❌ Error: GEMINI_API_KEY not found in .env file.")

## Step 1: Text Embeddings
Let's see how a text sentence is converted into a vector (a list of floating-point numbers). The similarity of these vectors determines how closely related the sentences are semantically.

In [ ]:
# Generate embeddings for a sample text
sample_text = "Operating margin for the quarter contracted to 25.2% due to elevated supply chain costs."

response = client.models.embed_content(
    model="text-embedding-004",
    contents=sample_text
)

vector = response.embeddings[0].values
print(f"Text: '{sample_text}'")
print(f"Vector Dimension: {len(vector)} dimensions")
print(f"Vector Snippet (first 10 numbers): {vector[:10]}")

## Step 2: Vector Database (ChromaDB) Querying
Now let's check how we search our existing database collections (`pdf_chunks` and `audio_chunks`) that were created during your ingestion step.

In [ ]:
# Connect to ChromaDB
chroma_client = chromadb.PersistentClient(path="chroma_db")
pdf_collection = chroma_client.get_collection("pdf_chunks")
audio_collection = chroma_client.get_collection("audio_chunks")

print("PDF Collection Size:", pdf_collection.count(), "chunks")
print("Audio Collection Size:", audio_collection.count(), "chunks")

Let's perform a semantic search. We will search for a query and see which chunks get returned with their cosine similarity distance scores (lower distance means higher similarity).

In [ ]:
query = "operating expenses and margins"
query_emb = client.models.embed_content(model="text-embedding-004", contents=query).embeddings[0].values

# Search PDF
pdf_results = pdf_collection.query(
    query_embeddings=[query_emb],
    where={"$and": [{"company": "Apple"}, {"quarter": "Q3-2024"}]},
    n_results=2
)

print(f"=== PDF Search Results for '{query}' ===")
for i, (doc, dist) in enumerate(zip(pdf_results['documents'][0], pdf_results['distances'][0])):
    print(f"Match {i+1} (Distance: {dist:.4f}):")
    print(doc)
    print("-" * 40)

## Step 3: LLM Augmentation & Generation
Let's merge the retrieved text with our audit prompt and see how Gemini processes both to find discrepancies.

In [ ]:
# Retrieve context from both PDF and Audio
pdf_res = pdf_collection.query(query_embeddings=[query_emb], n_results=2)
audio_res = audio_collection.query(query_embeddings=[query_emb], n_results=2)

pdf_context = "\n\n".join(pdf_res['documents'][0])
audio_context = "\n\n".join(audio_res['documents'][0])

# Prompt template (Prompt Engineering)
prompt = f"""You are a forensic auditor. Compare the two sources and note any contradictions:

=== PDF Filing ===
{pdf_context}

=== Audio Transcript ===
{audio_context}

Generate a summary and point out direct contradictions or sentiment shifts."""

# Call Gemini
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents=prompt
)

print("=== GEMINI AUDIT REPORT ===")
print(response.text)